# Generation Data Loading Test
This notebook tests the loading of the generation dataset and inspects the data format.

In [1]:
import sys
import os
import pprint
import torch
from transformers import AutoTokenizer

# Add root directory to path to import project modules
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_dir not in sys.path:
    sys.path.append(root_dir)

from ZGeneration.data_loader_gen import GenerationDataset, load_gen_data
from config_llama3 import TrainingConfig

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration
data_path = os.path.join(root_dir, 'data', 'gen_task')
print(f"Data Path: {data_path}")

# Load Tokenizer
# We try to load the tokenizer defined in config, or fallback to a default if unavailable
model_name = TrainingConfig.model_name
print(f"Model Name from Config: {model_name}")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("Tokenizer loaded successfully.")
except Exception as e:
    print(f"Warning: Could not load tokenizer '{model_name}'.")
    print(f"Error: {e}")
    print("Setting tokenizer to None. Dataset __getitem__ will fail, but raw data inspection is possible.")
    tokenizer = None

Data Path: /home/new/Documents/QSH/Llama_train/data/gen_task
Model Name from Config: Llama-3.3-8B-Instruct
Error: Llama-3.3-8B-Instruct is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`
Setting tokenizer to None. Dataset __getitem__ will fail, but raw data inspection is possible.


In [3]:
# Load Data
try:
    data, max_len = load_gen_data(data_path)
    print(f"Max Length in Data: {max_len}")
    
    # Initialize Dataset
    # pass max_seq_len from config or default
    dataset = GenerationDataset(data, tokenizer, max_seq_len=TrainingConfig.hidden_size if hasattr(TrainingConfig, 'hidden_size') else 512)
    print(f"Dataset Initialized. Size: {len(dataset)}")
    
except Exception as e:
    print(f"Error loading data: {e}")
    # Create dummy data for visualization if loading fails? No, better to stop.
    raise e

Loading Gen Data: /home/new/Documents/QSH/Llama_train/data/gen_task/processed_gen_data.pkl
Data Loaded: 51239 samples.
Max Length in Data: 2195
Dataset Initialized. Size: 51239


In [4]:
# Inspect First 10 Samples
print("Inspecting first 10 samples from the dataset (Raw Data)...\n")

for i in range(10):
    if i >= len(dataset):
        break
        
    print(f"====== Sample {i+1} ======")
    
    # Access raw data directly to avoid tokenization errors if tokenizer is missing
    # dataset.data is the dictionary structure
    input_text = dataset.data['input_text'][i]
    target_text = dataset.data['target_text'][i]
    emotion = dataset.data['emotion'][i]
    unique_idx = dataset.data['ud_idx'][i]
    local_idx = dataset.data['ld_idx'][i]
    
    print(f"Emotion: {emotion}")
    print(f"Dialogue ID: {unique_idx}, Turn ID: {local_idx}")
    print("\n[Input (Messages)]:")
    pprint.pprint(input_text, width=120)
    
    print("\n[Target Output]:")
    pprint.pprint(target_text)
    
    print("\n")

Inspecting first 10 samples from the dataset (Raw Data)...

====== Sample 1 ======
Emotion: sentimental
Dialogue ID: 0, Turn ID: 0

[Input (Messages)]:
[{'content': "You are an empathetic assistant. Your task is to understand the user's situation and feelings, and "
             'respond supportively.',
  'role': 'system'},
 {'content': 'Situation: i remember going to the fireworks with my best friend . there was a lot of people , but it '
             'only felt like us in the world .\n'
             '\n'
             'User Word: i remember going to see the fireworks with my best friend . it was the first time we ever '
             'spent time alone together . although there was a lot of people , we felt like the only people in the '
             'world .',
  'role': 'user'},
 {'content': 'was this a friend you were in love with , or just a best friend ?', 'role': 'assistant'}]

[Target Output]:
'was this a friend you were in love with , or just a best friend ?'


====== Sample 2 ===